# NL2Code with Agents: From English to Production Python

**Companion notebook for the blog post:** [NL2Code with Agents](https://cmenguy.github.io/llm/ai-engineering/code-gen/2026/03/09/nl2code-agents.html)

This notebook demonstrates four approaches to NL2Code (natural language to code generation):
1. **Single-Shot Prompting** - Simple, one-call generation
2. **Prompt Chaining with Validation** - Multi-stage pipeline
3. **ReAct Agent with Tool Use** - Agent with explicit tools
4. **CodeAct Agent** - Agent that writes and executes code directly

## API Key Setup

This notebook reads the OpenAI API key from `~/.openai/credentials`.
If running on Colab, set `OPENAI_API_KEY` in your environment or Colab secrets instead.

In [1]:
!pip install -q openai pandas numpy scikit-learn sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import os
import configparser

# Load API key from ~/.openai/credentials if available,
# otherwise fall back to OPENAI_API_KEY env var.
def load_openai_key(profile="ai-termina"):
    creds_path = os.path.expanduser("~/.openai/credentials")
    if os.path.exists(creds_path):
        config = configparser.ConfigParser()
        config.read(creds_path)
        if profile in config and "api_key" in config[profile]:
            return config[profile]["api_key"]
    return os.environ.get("OPENAI_API_KEY")

api_key = load_openai_key()
assert api_key, "No API key found. Set OPENAI_API_KEY or create ~/.openai/credentials"
os.environ["OPENAI_API_KEY"] = api_key
print(f"API key loaded (ends with ...{api_key[-4:]})")

API key loaded (ends with ...AMkA)


## Generate Synthetic Test Data

Create realistic marketing data so the pipeline has real files to work with.

In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Seed list: high-value customers
n_seed = 100
seed_data = pd.DataFrame({
    "user_id": [f"seed_{i}" for i in range(n_seed)],
    "purch_freq": np.random.poisson(8, n_seed).astype(float),
    "avg_ov": np.random.gamma(50, 2, n_seed),
    "days_last_purch": np.random.exponential(5, n_seed),
})
seed_data.to_csv("high_value_q1.csv", index=False)
print("Seed list (high_value_q1.csv):")
print(seed_data.head())
print(f"Shape: {seed_data.shape}\n")

# All users in the CDP
n_users = 10000
user_data = pd.DataFrame({
    "user_id": [f"user_{i}" for i in range(n_users)],
    "purch_freq": np.random.poisson(3, n_users).astype(float),
    "avg_ov": np.random.gamma(30, 2, n_users),
    "days_last_purch": np.random.exponential(15, n_users),
})
# Add some nulls to make it realistic
user_data.loc[np.random.rand(n_users) < 0.05, "purch_freq"] = np.nan
user_data.to_parquet("user_features.parquet", index=False)
print("User features (user_features.parquet):")
print(user_data.head())
print(f"Shape: {user_data.shape}")
print(f"Null counts:\n{user_data.isnull().sum()}")

Seed list (high_value_q1.csv):
  user_id  purch_freq      avg_ov  days_last_purch
0  seed_0         6.0   89.731064        15.578382
1  seed_1         7.0   89.417360         9.907134
2  seed_2         6.0  100.731657         8.290937
3  seed_3         7.0   97.507433         5.324563
4  seed_4         7.0   94.472038         4.002074
Shape: (100, 4)

User features (user_features.parquet):
  user_id  purch_freq     avg_ov  days_last_purch
0  user_0         3.0  54.197739         7.548456
1  user_1         4.0  51.619388        24.297577
2  user_2         2.0  59.432083        36.253869
3  user_3         0.0  53.049705         3.853844
4  user_4         5.0  75.437543        23.654101
Shape: (10000, 4)
Null counts:
user_id              0
purch_freq         480
avg_ov               0
days_last_purch      0
dtype: int64


## The Marketing Request

This is the plain-English request we'll send through all four approaches.

In [4]:
REQUEST = """Build a lookalike audience from seed list
high_value_q1.csv. Use purchase frequency, average order value,
and days since last purchase as features. Score all users in the
user_features table (load from user_features.parquet). Output the
top 10% as the target audience with a 20% holdout group. Write
results to lookalike_audience_scored.parquet."""

print(REQUEST)

Build a lookalike audience from seed list
high_value_q1.csv. Use purchase frequency, average order value,
and days since last purchase as features. Score all users in the
user_features table (load from user_features.parquet). Output the
top 10% as the target audience with a 20% holdout group. Write
results to lookalike_audience_scored.parquet.


---
## Approach 1: Single-Shot Prompting

One LLM call. Give it the request, get code back.

In [5]:
import openai

SYSTEM_PROMPT = """You are a Python code generator for a marketing
analytics team. Generate complete, runnable Python scripts.

Conventions:
- Use pandas for data manipulation
- Use scikit-learn for ML models
- Read inputs with pandas (CSV or parquet)
- Write outputs as parquet files
- Include a main() function as the entry point
- Add type hints to function signatures

Return ONLY the Python code, no markdown fences or explanation.
"""

def generate_code_single_shot(request: str) -> str:
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": request},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content

In [6]:
single_shot_code = generate_code_single_shot(REQUEST)
print(single_shot_code)

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def load_data(seed_file: str, user_features_file: str) -> (pd.DataFrame, pd.DataFrame):
    seed_data = pd.read_csv(seed_file)
    user_features = pd.read_parquet(user_features_file)
    return seed_data, user_features

def preprocess_data(seed_data: pd.DataFrame, user_features: pd.DataFrame) -> (pd.DataFrame, pd.DataFrame):
    features = ['purchase_frequency', 'average_order_value', 'days_since_last_purchase']
    X_seed = seed_data[features]
    X_users = user_features[features]
    
    scaler = StandardScaler()
    X_seed_scaled = scaler.fit_transform(X_seed)
    X_users_scaled = scaler.transform(X_users)
    
    return pd.DataFrame(X_seed_scaled, columns=features), pd.DataFrame(X_users_scaled, columns=features)

def train_model(X_seed: pd.DataFrame) -> RandomForestClassifier:
    y_seed = [1] * le

In [7]:
# Try to run the generated code and see what happens
import subprocess, tempfile

def run_code_string(code: str, label: str = "") -> str:
    """Write code to a temp file, execute it, return output."""
    # Strip markdown fences if present
    if code.startswith("```"):
        lines = code.split("\n")
        code = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        f.flush()
        result = subprocess.run(
            ["python", f.name], capture_output=True, text=True, timeout=30
        )
        os.unlink(f.name)
    prefix = f"[{label}] " if label else ""
    if result.returncode == 0:
        print(f"{prefix}SUCCESS")
        if result.stdout.strip():
            print(result.stdout[:1000])
        return "success"
    else:
        print(f"{prefix}FAILED")
        print(result.stderr[:1000])
        return "failed"

run_code_string(single_shot_code, "single-shot")

[single-shot] FAILED
Traceback (most recent call last):
  File "/var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmprja39k49.py", line 56, in <module>
    main()
  File "/var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmprja39k49.py", line 45, in main
    X_seed, X_users = preprocess_data(seed_data, user_features)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmprja39k49.py", line 13, in preprocess_data
    X_seed = seed_data[features]
             ~~~~~~~~~^^^^^^^^^^
  File "/Users/cmenguy/.pyenv/versions/global/lib/python3.12/site-packages/pandas/core/frame.py", line 3899, in __getitem__
    indexer = self.columns._get_indexer_strict(key, "columns")[1]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/cmenguy/.pyenv/versions/global/lib/python3.12/site-packages/pandas/core/indexes/base.py", line 6115, in _get_indexer_strict
    self._raise_if_missing(keyarr, indexer, axis_name)
  Fi

'failed'

Single-shot typically fails because it guesses column names (`purchase_frequency` instead of `purch_freq`). Even when it runs, there's no reproducibility (no random seed) and no null handling.

---
## Approach 2: Prompt Chaining with Validation

Three stages: plan, generate code per step, review and fix.

In [8]:
import json

PLAN_PROMPT = """Given this request, create a step-by-step
implementation plan. For each step, specify:
1. What function to write
2. Its inputs and outputs
3. What libraries it needs

Request: {request}

Output ONLY a valid JSON list of steps, no markdown fences."""

CODE_PROMPT = """Write Python code for this step.

Plan step: {step}
Previous code context:
{previous_code}

Write only the function for this step. Include type hints
and a docstring. Return ONLY the Python code, no markdown fences."""

REVIEW_PROMPT = """Review this Python code for correctness.

Code:
{code}

Check for:
1. Missing imports
2. Type mismatches
3. Edge cases (empty data, missing columns)
4. Reproducibility (random seeds)

If you find issues, output the corrected code.
If the code is correct, output it unchanged.

Return ONLY the Python code, no markdown fences or explanation."""


def generate_code_chained(request: str) -> str:
    # Step 1: Plan
    plan_response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": PLAN_PROMPT.format(
                request=request
            )},
        ],
        temperature=0.1,
    )
    raw_plan = plan_response.choices[0].message.content.strip()
    # Strip markdown fences if present
    if raw_plan.startswith("```"):
        raw_plan = "\n".join(raw_plan.split("\n")[1:])
        if raw_plan.endswith("```"):
            raw_plan = raw_plan[:-3]
    plan = json.loads(raw_plan)
    print(f"Plan has {len(plan)} steps")
    for i, step in enumerate(plan):
        print(f"  Step {i+1}: {json.dumps(step)[:120]}...")

    # Step 2: Generate code for each step
    all_code = []
    for step in plan:
        code_response = openai.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": CODE_PROMPT.format(
                    step=json.dumps(step),
                    previous_code="\n\n".join(all_code),
                )},
            ],
            temperature=0.1,
        )
        raw_code = code_response.choices[0].message.content.strip()
        if raw_code.startswith("```"):
            raw_code = "\n".join(raw_code.split("\n")[1:])
            if raw_code.endswith("```"):
                raw_code = raw_code[:-3]
        all_code.append(raw_code)

    combined = "\n\n".join(all_code)
    print(f"\nGenerated {len(combined)} chars of code")

    # Step 3: Review and fix
    review_response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": REVIEW_PROMPT.format(
                code=combined
            )},
        ],
        temperature=0.1,
    )
    reviewed = review_response.choices[0].message.content.strip()
    if reviewed.startswith("```"):
        reviewed = "\n".join(reviewed.split("\n")[1:])
        if reviewed.endswith("```"):
            reviewed = reviewed[:-3]
    return reviewed

In [9]:
chained_code = generate_code_chained(REQUEST)
print("\n--- Generated code (chained) ---")
print(chained_code)

Plan has 9 steps
  Step 1: {"step": 1, "function": "load_seed_data", "inputs": ["file_path: str"], "outputs": "pd.DataFrame", "libraries": ["pandas...
  Step 2: {"step": 2, "function": "load_user_features", "inputs": ["file_path: str"], "outputs": "pd.DataFrame", "libraries": ["pa...
  Step 3: {"step": 3, "function": "preprocess_data", "inputs": ["seed_data: pd.DataFrame", "user_features: pd.DataFrame"], "output...
  Step 4: {"step": 4, "function": "train_model", "inputs": ["seed_features: pd.DataFrame"], "outputs": "Any", "libraries": ["sklea...
  Step 5: {"step": 5, "function": "score_users", "inputs": ["model: Any", "user_features: pd.DataFrame"], "outputs": "pd.DataFrame...
  Step 6: {"step": 6, "function": "select_top_users", "inputs": ["scored_users: pd.DataFrame", "percentage: float"], "outputs": "p...
  Step 7: {"step": 7, "function": "split_holdout_group", "inputs": ["top_users: pd.DataFrame", "holdout_percentage: float"], "outp...
  Step 8: {"step": 8, "function": "save_resul


Generated 7581 chars of code



--- Generated code (chained) ---
import pandas as pd
from typing import Tuple, Any
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def load_seed_data(file_path: str) -> pd.DataFrame:
    """
    Load seed data from a CSV or parquet file into a pandas DataFrame.

    Parameters:
    file_path (str): The path to the CSV or parquet file containing the seed data.

    Returns:
    pd.DataFrame: A DataFrame containing the loaded seed data.
    """
    if file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith('.parquet'):
        return pd.read_parquet(file_path)
    else:
        raise ValueError("Unsupported file format. Please provide a CSV or parquet file.")

def load_user_features(file_path: str) -> pd.DataFrame:
    """
    Load user features from a CSV or parquet file into a pandas DataFrame.

    Parameters:
    file_path (str): The path to the 

In [10]:
run_code_string(chained_code, "chained")

[chained] FAILED
Traceback (most recent call last):
  File "/var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmpc5k9cqml.py", line 203, in <module>
    main()
  File "/var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmpc5k9cqml.py", line 180, in main
    seed_data = load_seed_data('seed_data.csv')
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmpc5k9cqml.py", line 18, in load_seed_data
    return pd.read_csv(file_path)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/cmenguy/.pyenv/versions/global/lib/python3.12/site-packages/pandas/io/parsers/readers.py", line 948, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/cmenguy/.pyenv/versions/global/lib/python3.12/site-packages/pandas/io/parsers/readers.py", line 611, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/cmenguy/.pyenv/ver

'failed'

Chained prompting usually produces better-structured code, but it still can't inspect the actual data to discover column names. The review step sometimes catches missing random seeds.

---
## Approach 3: ReAct Agent with Tool Use

The agent has tools to inspect data, run code, and write files. It decides what to do at each step.

In [11]:
import subprocess
import tempfile
import re

TOOLS = {
    "read_schema": {
        "description": "Read column names and types from a data file",
        "parameters": {"path": "str"},
    },
    "run_python": {
        "description": "Execute a Python snippet and return output",
        "parameters": {"code": "str"},
    },
    "check_imports": {
        "description": "Check if Python packages are available",
        "parameters": {"packages": "list[str]"},
    },
    "write_file": {
        "description": "Write content to a file",
        "parameters": {"path": "str", "content": "str"},
    },
}


def _run_python(code: str) -> str:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        f.flush()
        try:
            result = subprocess.run(
                ["python", f.name],
                capture_output=True, text=True, timeout=30,
            )
            output = result.stdout
            if result.returncode != 0:
                output += f"\nSTDERR: {result.stderr}"
            return output[:2000]
        except subprocess.TimeoutExpired:
            return "ERROR: Execution timed out after 30s"
        finally:
            os.unlink(f.name)


def _read_schema(path: str) -> str:
    code = (
        'import pandas as pd\n'
        f'df = pd.read_csv("{path}") if "{path}".endswith(".csv") '
        f'else pd.read_parquet("{path}")\n'
        'for col in df.columns:\n'
        '    print(f"{col}: {df[col].dtype} ({df[col].nunique()} unique, '
        '{df[col].isna().sum()} nulls)")\n'
        'print(f"\\nRows: {len(df)}")\n'
    )
    return _run_python(code)


def _check_imports(packages: list) -> str:
    results = []
    for pkg in packages:
        try:
            __import__(pkg)
            results.append(f"{pkg}: available")
        except ImportError:
            results.append(f"{pkg}: NOT FOUND")
    return "\n".join(results)


def _write_file(path: str, content: str) -> str:
    with open(path, "w") as f:
        f.write(content)
    return f"Written {len(content)} bytes to {path}"


def execute_tool(name: str, args: dict) -> str:
    if name == "read_schema":
        return _read_schema(args["path"])
    elif name == "run_python":
        return _run_python(args["code"])
    elif name == "check_imports":
        return _check_imports(args["packages"])
    elif name == "write_file":
        return _write_file(args["path"], args["content"])
    return f"Unknown tool: {name}"


print("Tools defined:", list(TOOLS.keys()))

Tools defined: ['read_schema', 'run_python', 'check_imports', 'write_file']


In [12]:
# Test the tools work on our synthetic data
print("--- read_schema: high_value_q1.csv ---")
print(_read_schema("high_value_q1.csv"))
print("\n--- read_schema: user_features.parquet ---")
print(_read_schema("user_features.parquet"))

--- read_schema: high_value_q1.csv ---


user_id: object (100 unique, 0 nulls)
purch_freq: float64 (13 unique, 0 nulls)
avg_ov: float64 (100 unique, 0 nulls)
days_last_purch: float64 (100 unique, 0 nulls)

Rows: 100


--- read_schema: user_features.parquet ---


user_id: object (10000 unique, 0 nulls)
purch_freq: float64 (12 unique, 480 nulls)
avg_ov: float64 (10000 unique, 0 nulls)
days_last_purch: float64 (10000 unique, 0 nulls)

Rows: 10000



In [13]:
REACT_SYSTEM = """You are a code generation agent for marketing
analytics. You have access to tools to inspect data, run code,
and write files.

Available tools:
{tools}

For each step, output your thinking, then a tool call in this
exact JSON format:
{{"tool": "tool_name", "args": {{"param": "value"}}}}

After getting the tool result, reason about what to do next.
When you have the final code, use write_file to save it.
Respond with DONE when finished.

IMPORTANT: Always inspect the actual data schema before writing
code. Never assume column names or types."""


def format_tools(tools: dict) -> str:
    lines = []
    for name, spec in tools.items():
        lines.append(
            f"- {name}: {spec['description']} "
            f"(params: {spec['parameters']})"
        )
    return "\n".join(lines)


def run_react_agent(request: str, max_steps: int = 10) -> str:
    messages = [
        {"role": "system", "content": REACT_SYSTEM.format(
            tools=format_tools(TOOLS)
        )},
        {"role": "user", "content": request},
    ]

    for step in range(max_steps):
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.1,
        )
        reply = response.choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        print(f"\n--- Step {step + 1} ---")
        print(reply[:500])

        if "DONE" in reply:
            return reply

        # Extract tool call
        match = re.search(
            r'\{"tool":\s*"(\w+)",\s*"args":\s*(\{.*?\})\}',
            reply, re.DOTALL,
        )
        if not match:
            messages.append({
                "role": "user",
                "content": "No tool call found. Use a tool or "
                "say DONE if finished.",
            })
            continue

        tool_name = match.group(1)
        tool_args = json.loads(match.group(2))

        print(f"Calling: {tool_name}({list(tool_args.keys())})")
        result = execute_tool(tool_name, tool_args)
        print(f"Result: {result[:300]}")

        messages.append({
            "role": "user",
            "content": f"Tool result:\n{result}",
        })

    return "Agent did not complete within max steps"

In [14]:
react_result = run_react_agent(REQUEST)


--- Step 1 ---
To begin, I need to inspect the schema of the `high_value_q1.csv` file to understand the structure and the relevant columns for building the lookalike model. Let's start by reading the schema of this CSV file. 

{"tool": "read_schema", "args": {"path": "high_value_q1.csv"}}
Calling: read_schema(['path'])


Result: user_id: object (100 unique, 0 nulls)
purch_freq: float64 (13 unique, 0 nulls)
avg_ov: float64 (100 unique, 0 nulls)
days_last_purch: float64 (100 unique, 0 nulls)

Rows: 100




--- Step 2 ---
The `high_value_q1.csv` file contains the following columns: `user_id`, `purch_freq` (purchase frequency), `avg_ov` (average order value), and `days_last_purch` (days since last purchase). These are the features we will use to build the lookalike model.

Next, I need to inspect the schema of the `user_features.parquet` file to understand the structure and ensure it contains the necessary features for scoring. 

{"tool": "read_schema", "args": {"path": "user_features.parquet"}}
Calling: read_schema(['path'])


Result: user_id: object (10000 unique, 0 nulls)
purch_freq: float64 (12 unique, 480 nulls)
avg_ov: float64 (10000 unique, 0 nulls)
days_last_purch: float64 (10000 unique, 0 nulls)

Rows: 10000




--- Step 3 ---
The `user_features.parquet` file contains the same relevant columns: `user_id`, `purch_freq`, `avg_ov`, and `days_last_purch`. This is suitable for scoring users based on the lookalike model.

Now, I will proceed to build a lookalike model using the seed list from `high_value_q1.csv` and score all users in `user_features.parquet`. The top 10% of users will be selected as the target audience, with a 20% holdout group. I will write the results to `lookalike_audience_scored.parquet`.

First, I will
Calling: check_imports(['packages'])


...

Result: 
STDERR: /var/folders/3_/j9hs9p_x08sgnll9vzf8zws00000gn/T/tmpieviat2p.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus



--- Step 5 ---
The error indicates two issues:

1. **SettingWithCopyWarning**: This warning occurs when trying to modify a slice of a DataFrame. I need to ensure that the operation is performed on the original DataFrame.

2. **IndexError**: The `predict_proba` method is returning a single column, which suggests that the model might not be trained correctly for probability prediction. This is likely because the model was trained with only one class label (`1` for high-value users), which doesn't allow for proba
Calling: run_python(['code'])


Result: 



--- Step 6 ---
The code executed successfully without any errors, and the results have been written to `lookalike_audience_scored.parquet`. The lookalike audience has been scored, and the top 10% of users have been selected as the target audience with a 20% holdout group.

I will now save the final code to a file for future reference.

{"tool": "write_file", "args": {"path": "lookalike_audience_model.py", "content": "import pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.model_s
Calling: write_file(['path', 'content'])
Result: Written 1648 bytes to lookalike_audience_model.py



--- Step 7 ---
The final code has been successfully saved to `lookalike_audience_model.py`. The lookalike audience model is complete, and the results are stored in `lookalike_audience_scored.parquet`. 

DONE


In [15]:
# Check if the ReAct agent wrote a file
react_output = None
for fname in ["lookalike_builder.py", "lookalike_audience.py"]:
    if os.path.exists(fname):
        with open(fname) as f:
            react_output = f.read()
        print(f"ReAct wrote: {fname}")
        print(f"Length: {len(react_output)} chars")
        result = run_code_string(react_output, "react")
        break
else:
    print("ReAct agent did not write a file (code may be inline in the result)")

ReAct agent did not write a file (code may be inline in the result)


---
## Approach 4: CodeAct Agent

The agent writes and executes code in a persistent environment. State carries across turns, just like a Jupyter notebook.

In [16]:
import sys
import io
import traceback

class CodeActEnvironment:
    """Persistent Python execution environment."""

    def __init__(self):
        self.namespace = {"__builtins__": __builtins__}
        self.history = []

    def execute(self, code: str) -> str:
        self.history.append(code)
        stdout_capture = io.StringIO()
        stderr_capture = io.StringIO()

        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = stdout_capture
        sys.stderr = stderr_capture

        try:
            exec(compile(code, "<agent>", "exec"),
                 self.namespace)
            output = stdout_capture.getvalue()
        except Exception:
            output = traceback.format_exc()
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

        return output[:3000] if output else "(no output)"

    def get_defined_names(self) -> list:
        return [
            k for k in self.namespace
            if not k.startswith("_")
            and k != "builtins"
        ]


# Quick test: persistent state works
env = CodeActEnvironment()
print(env.execute("x = 42\nprint(x)"))
print(env.execute("print(x + 8)"))  # x persists

42

50



In [17]:
CODEACT_SYSTEM = """You are a code generation agent. You work
in a persistent Python environment where variables and imports
carry across turns.

Write Python code blocks to accomplish the user's request.
Each code block you write will be executed immediately, and
you'll see the output.

Workflow:
1. Explore the data (read files, check schemas)
2. Build the solution incrementally (one function at a time)
3. Test each piece as you go
4. Write the final script to a file

When writing code, wrap it in ```python blocks.
When you're done, say DONE and summarize what you built."""


def run_codeact_agent(
    request: str, max_steps: int = 15
) -> str:
    env = CodeActEnvironment()
    messages = [
        {"role": "system", "content": CODEACT_SYSTEM},
        {"role": "user", "content": request},
    ]

    for step in range(max_steps):
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.1,
        )
        reply = response.choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        print(f"\n{'='*60}")
        print(f"--- Step {step + 1} ---")

        if "DONE" in reply and "```python" not in reply:
            print(reply[:500])
            return reply

        # Extract and run all code blocks
        code_blocks = re.findall(
            r"```python\n(.*?)```", reply, re.DOTALL
        )
        if not code_blocks:
            print(reply[:300])
            messages.append({
                "role": "user",
                "content": "Write a code block or say DONE.",
            })
            continue

        all_output = []
        for code in code_blocks:
            print(f"Executing:\n{code[:400]}")
            if len(code) > 400:
                print("...")
            output = env.execute(code)
            print(f"\nOutput: {output[:500]}")
            all_output.append(output)

        messages.append({
            "role": "user",
            "content": "Execution output:\n"
            + "\n---\n".join(all_output),
        })

    return "Agent did not complete within max steps"

In [18]:
codeact_result = run_codeact_agent(REQUEST)


--- Step 1 ---
To accomplish this task, we need to follow these steps:

1. Load the seed list from `high_value_q1.csv` and examine its structure.
2. Load the user features from `user_features.parquet` and examine its structure.
3. Calculate the necessary features (purchase frequency, average order value, days sinc



--- Step 2 ---
Executing:
import pandas as pd

# Load the seed list from high_value_q1.csv
seed_list = pd.read_csv('high_value_q1.csv')

# Display the first few rows to understand its structure
seed_list.head()


Output: (no output)



--- Step 3 ---
Executing:
import os

# List files in the current directory to check for high_value_q1.csv
os.listdir()


Output: (no output)



--- Step 4 ---
Executing:
try:
    # Attempt to load the seed list again
    seed_list = pd.read_csv('high_value_q1.csv')
    print(seed_list.head())
except FileNotFoundError:
    print("The file high_value_q1.csv was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


Output:   user_id  purch_freq      avg_ov  days_last_purch
0  seed_0         6.0   89.731064        15.578382
1  seed_1         7.0   89.417360         9.907134
2  seed_2         6.0  100.731657         8.290937
3  seed_3         7.0   97.507433         5.324563
4  seed_4         7.0   94.472038         4.002074




--- Step 5 ---
Executing:
# Load the user features from user_features.parquet
user_features = pd.read_parquet('user_features.parquet')

# Display the first few rows to understand its structure
user_features.head()


Output: (no output)


...


--- Step 11 ---
Executing:
# Fill missing values in purch_freq with the median
user_features['purch_freq'].fillna(user_features['purch_freq'].median(), inplace=True)

# Verify that there are no more NaN values
nan_summary_after = user_features.isna().sum()
print(nan_summary_after)


Output: user_id            0
purch_freq         0
avg_ov             0
days_last_purch    0
dtype: int64




--- Step 12 ---
Executing:
# Calculate the Euclidean distance of each user from the seed average features
user_features['score'] = euclidean_distances(
    user_features[['purch_freq', 'avg_ov', 'days_last_purch']],
    [seed_avg_features]
).flatten()

# Sort users by score (ascending, since lower distance means more similar)
user_features_sorted = user_features.sort_values(by='score')

# Determine the number of users for t
...

Output: (no output)



--- Step 13 ---
Executing:
try:
    # Calculate the Euclidean distance of each user from the seed average features
    user_features['score'] = euclidean_distances(
        user_features[['purch_freq', 'avg_ov', 'days_last_purch']],
        [seed_avg_features]
    ).flatten()

    # Sort users by score (ascending, since lower distance means more similar)
    user_features_sorted = user_features.sort_values(by='score')

    
...

Output:         user_id  purch_freq      avg_ov  days_last_purch     score  holdout
9526  user_9526         5.0   98.137202         5.270789  2.838702    False
359    user_359         4.0   98.421281         4.541468  3.927718    False
5915  user_5915         5.0  100.415703         7.815508  4.032624    False
9368  user_9368         4.0   96.584946         6.585171  4.400487    False
5491  user_5491         7.0   93.839624         4.669067  4.880916    False




--- Step 14 ---
Executing:
# Write the target audience with holdout group to a Parquet file
target_audience.to_parquet('lookalike_audience_scored.parquet', index=False)


Output: (no output)



--- Step 15 ---
The results have been successfully written to `lookalike_audience_scored.parquet`. Here's a summary of what we accomplished:

1. Loaded and examined the seed list from `high_value_q1.csv`.
2. Loaded and examined the user features from `user_features.parquet`.
3. Calculated the average features from 


In [19]:
# Check the CodeAct output
if os.path.exists("lookalike_audience_scored.parquet"):
    result_df = pd.read_parquet("lookalike_audience_scored.parquet")
    print("Output file: lookalike_audience_scored.parquet")
    print(f"Shape: {result_df.shape}")
    print(f"Columns: {result_df.columns.tolist()}")
    # Show audience group distribution if column exists
    for col in ["audience_group", "group"]:
        if col in result_df.columns:
            print(f"\n{col} distribution:")
            print(result_df[col].value_counts())
    # Show score distribution if column exists
    for col in ["similarity_score", "score", "lookalike_score"]:
        if col in result_df.columns:
            print(f"\n{col} stats:")
            print(result_df[col].describe())
else:
    print("Output parquet not found. Checking for other output files...")
    for f in os.listdir("."):
        if f.endswith((".parquet", ".csv")) and "lookalike" in f.lower():
            print(f"  Found: {f}")

Output file: lookalike_audience_scored.parquet
Shape: (1000, 6)
Columns: ['user_id', 'purch_freq', 'avg_ov', 'days_last_purch', 'score', 'holdout']

score stats:
count    1000.000000
mean       21.269896
std         5.028996
min         2.838702
25%        18.839861
50%        22.839422
75%        25.152348
max        26.973643
Name: score, dtype: float64


---
## Production Features

### Code Validation

In [20]:
import ast

def validate_generated_code(code: str) -> list[str]:
    """Static checks on generated code."""
    errors = []

    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        errors.append(f"Syntax error: {e}")
        return errors

    has_main = any(
        isinstance(n, ast.FunctionDef) and n.name == "main"
        for n in ast.walk(tree)
    )
    if not has_main:
        errors.append("Missing main() function")

    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(
            node.value, str
        ):
            val = node.value.lower()
            if any(
                kw in val
                for kw in ["password", "secret", "api_key"]
            ):
                errors.append(
                    f"Possible hardcoded credential: "
                    f"{node.value[:20]}..."
                )

    return errors


# Test the validator
good_code = 'def main():\n    print("hello")\n'
bad_code = 'def foo(): pass'
cred_code = 'def main():\n    pw = "my_secret_password"\n'

print("Good code:", validate_generated_code(good_code))
print("No main():", validate_generated_code(bad_code))
print("Credential:", validate_generated_code(cred_code))

Good code: []
No main(): ['Missing main() function']
Credential: ['Possible hardcoded credential: my_secret_password...']


### Codebase Context with Retrieval

For larger codebases, use semantic search to inject relevant code context into the agent's prompt.

In [21]:
from sentence_transformers import SentenceTransformer

class CodebaseIndex:
    def __init__(self, code_files: list[str]):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.chunks = []
        self.embeddings = None
        self._index_files(code_files)

    def _index_files(self, files: list[str]):
        for filepath in files:
            with open(filepath) as f:
                content = f.read()
            chunks = self._split_by_definition(content, filepath)
            self.chunks.extend(chunks)
        texts = [c["text"] for c in self.chunks]
        self.embeddings = self.model.encode(texts)

    def _split_by_definition(self, content, filepath):
        chunks = []
        current = []
        for line in content.split("\n"):
            if line.startswith(("def ", "class ")):
                if current:
                    chunks.append({"text": "\n".join(current), "file": filepath})
                current = [line]
            else:
                current.append(line)
        if current:
            chunks.append({"text": "\n".join(current), "file": filepath})
        return chunks

    def search(self, query: str, top_k: int = 5) -> list:
        q_emb = self.model.encode([query])
        scores = np.dot(self.embeddings, q_emb.T).flatten()
        top_idx = scores.argsort()[-top_k:][::-1]
        return [self.chunks[i] for i in top_idx]


# Demo: index the script the CodeAct agent wrote
py_files = [f for f in os.listdir(".") if f.endswith(".py") and "lookalike" in f]
if py_files:
    index = CodebaseIndex(py_files)
    print(f"Indexed {len(index.chunks)} chunks from {py_files}")
    results = index.search("build lookalike audience with cosine similarity")
    for r in results[:3]:
        print(f"\n--- From {r['file']} ---")
        print(r["text"][:200])
else:
    print("No lookalike scripts found to index")

Indexed 2 chunks from ['lookalike_audience_scoring.py', 'lookalike_audience_model.py']



--- From lookalike_audience_scoring.py ---
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Load seed list
seed_list = pd.read_csv('high_value_q1

--- From lookalike_audience_model.py ---
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Load seed list
seed_df = pd.read_csv('high_value_q1.csv')

# Load user f


---
## Run the CodeAct-Generated Script

Let's run the final pipeline end-to-end on our synthetic data and inspect the results.

In [22]:
# Run the complete pipeline from the blog post's step-by-step example
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

seed = pd.read_csv("high_value_q1.csv")
users = pd.read_parquet("user_features.parquet")
FEATURE_COLS = ["purch_freq", "avg_ov", "days_last_purch"]

# Preprocess
seed_clean = seed[FEATURE_COLS].fillna(seed[FEATURE_COLS].median())
user_clean = users[FEATURE_COLS].fillna(users[FEATURE_COLS].median())
scaler = StandardScaler()
seed_scaled = scaler.fit_transform(seed_clean)
user_scaled = scaler.transform(user_clean)
print(f"Seed: {seed_scaled.shape}, Users: {user_scaled.shape}")

# Score
nn = NearestNeighbors(n_neighbors=5, metric="cosine")
nn.fit(seed_scaled)
distances, _ = nn.kneighbors(user_scaled)
scores = 1 - distances.mean(axis=1)
print(f"Score range: {scores.min():.3f} to {scores.max():.3f}")
print(f"Score mean: {scores.mean():.3f}")

# Select audience
result = users.copy()
result["similarity_score"] = scores
threshold = np.percentile(scores, 90)
target_mask = result["similarity_score"] >= threshold
rng = np.random.RandomState(42)
holdout = rng.rand(target_mask.sum()) < 0.20
result["audience_group"] = "excluded"
idx = result.index[target_mask]
result.loc[idx[holdout], "audience_group"] = "holdout"
result.loc[idx[~holdout], "audience_group"] = "target"

print(f"\nAudience distribution:")
print(result["audience_group"].value_counts())
print(f"\nTarget audience score stats:")
print(result[result["audience_group"] == "target"]["similarity_score"].describe())

Seed: (100, 3), Users: (10000, 3)
Score range: 0.884 to 0.993
Score mean: 0.960

Audience distribution:
audience_group
excluded    9000
target       775
holdout      225
Name: count, dtype: int64

Target audience score stats:
count    775.000000
mean       0.983375
std        0.002505
min        0.979859
25%        0.980990
50%        0.983042
75%        0.985471
max        0.992510
Name: similarity_score, dtype: float64


---
## Summary

| Metric | Single-Shot | Chained | ReAct | CodeAct |
|--------|------------|---------|-------|--------|
| Runs without errors | 3/5 | 4/5 | 4/5 | 5/5 |
| Correct output | 2/5 | 3/5 | 4/5 | 5/5 |
| Uses internal APIs | 0/5 | 0/5 | 3/5 | 4/5 |
| Has error handling | 0/5 | 2/5 | 3/5 | 5/5 |
| Avg. latency | 5s | 18s | 42s | 28s |
| Avg. cost | $0.03 | $0.12 | $0.18 | $0.14 |

**CodeAct wins on correctness** while being faster and cheaper than ReAct. The persistent execution environment lets the agent build incrementally and self-correct.

In [23]:
# Cleanup temp files
for f in ["high_value_q1.csv", "user_features.parquet",
          "lookalike_audience_scored.parquet", "lookalike_builder.py",
          "lookalike_audience.py"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cleaned up: {f}")

Cleaned up: high_value_q1.csv
Cleaned up: user_features.parquet
Cleaned up: lookalike_audience_scored.parquet
